Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Additional Umbrella Sampling near RC~12 (with a stronger spring constant)

With a weak spring constant (`KAPPA=2.5 eV`, window spacing 0.2 A), the system can slip away from the restraint center, so the histogram overlap between neighboring windows may become insufficient.

In this notebook we run additional umbrella sampling, targeting only the region around 12 A, with

- a stronger spring constant `KAPPA` (default `2.5 -> 10.0 eV`)
- a finer window spacing (0.2 -> 0.1 A)

In the next notebook, `08_mbar_free_energy_with_extra_en.ipynb`, the original windows and these extra windows are analyzed together with MBAR.

## Step 1. Import libraries and configure the environment

The environment setup is the same as in `05_umbrella_sampling_en.ipynb`. We set the environment variables required to use PLUMED from Python.

In [ ]:
# Install the packages below if needed
#!pip install joblib
#!pip install plumed

In [ ]:
# ============================================================
# 1. Environment & Imports
# ============================================================

import os
import sys
import glob
import numpy as np
from time import perf_counter
from joblib import Parallel, delayed

# ASE
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.constraints import FixAtoms

# PLUMED wrapper
from ase.calculators.plumed import Plumed

# PFP (Matlantis)
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

# PLUMED Environment Variables (Please modify if necessary to match the path of your environment.)
plumed_path = "/home/jovyan/local/plumed-2.9.0"
os.environ["PLUMED_KERNEL"] = f"{plumed_path}/lib/libplumedKernel.so"
os.environ["PLUMED_TYPESAFE_IGNORE"] = "yes"
sys.path.append(plumed_path)

## Step 2. Set the calculation parameters

The MD temperature and number of steps are kept identical to the original calculation (so that they can be treated as the same temperature in the subsequent MBAR analysis).
The settings specific to the additional sampling are as follows.

* **KAPPA_STRONG**: spring constant for the extra windows. Stronger than the original `2.5 eV` (default `10.0 eV`).
* **cv_restraint_extra**: list of additional restraint centers. They are placed finely around 12 A in 0.1 A increments.
* **ORIG_US_DIR**: output directory of the original US run. The equilibrated structure of the nearest center is reused as the initial structure for each extra window.
* **OUT_BASE**: output destination for the extra windows.

In [ ]:
# ============================================================
# 2. Parameters & Settings
# ============================================================

# PFP Settings (keep identical to the original calculation)
CALC_MODE     = "PBE_PLUS_D3"
METHOD_TYPE   = "PFVM_D3_PFVM"
MODEL_VERSION = "v8.0.0"

# Joblib Settings (parallel computation settings)
N_JOBS  = 10           # Number of concurrent jobs (adjust to your environment)
VERBOSE = 10           # Verbosity of the progress display
BACKEND = "threading"  # "threading" is recommended when using Matlantis

# MD Settings (keep consistent with the original run: temperature, timestep, number of steps)
TEMPERATURE  = 375.0        # Kelvin
TIMESTEP     = 1.0 * units.fs
TOTAL_STEPS  = 50_000       # Number of steps per window
LOG_INTERVAL = 100

# ---- Settings specific to the additional sampling ----
# Spring constant for the extra windows (originally 2.5 eV; a stronger value keeps
# the system reliably near 12 A)
KAPPA_STRONG = 10.0         # eV

# Additional restraint centers (finely spaced around 12 A in 0.1 A increments)
cv_restraint_extra = np.round(np.arange(11.6, 12.41, 0.1), 2)

# Pulled atom (PLUMED is 1-based; corresponds to the Cu atom at ASE 0-based index 267)
PULLED_ATOM = 268
# Threshold for fixing the lower Cu layers (atoms with z < FIX_THRESH are fixed)
FIX_THRESH  = 4.0

# Input / output directories
# Original US output (source of initial structures); falls back to assets if not in output
ORIG_US_DIR = "./output/05_umbrella_sampling"
if not os.path.isdir(ORIG_US_DIR):
    ORIG_US_DIR = "./assets/05_umbrella_sampling"
    print(f"Directory not found in output, loading from assets: {ORIG_US_DIR}")
OUT_BASE    = "./output/07_umbrella_sampling_extra"    # Output for the extra windows (does not overwrite the original)

print(f"KAPPA (strong) : {KAPPA_STRONG} eV  (original 2.5 eV)")
print(f"Extra windows  : {len(cv_restraint_extra)}")
print(f"Values         : {cv_restraint_extra}")
print(f"Output base    : {OUT_BASE}")

## Step 3. Providing the initial structures

For each extra window, we reuse the equilibrated structure of the original US window whose restraint center is closest. Since that structure has already been stretched to near the target RC,
equilibration is fast and stable.

If the original US output (`output/05_umbrella_sampling/`) is not available, the initial structures extracted in 04 (`output/04_initial_structures/`) are used, and failing that the 0.1 Å-spaced initial structures shipped in `assets/05_umbrella_sampling/`, so this notebook can also be run on its own.

In [ ]:
# ============================================================
# 3. Initial structure provider
# ============================================================
# Get the list of centers of the original US windows (empty if the output of 05 does not exist)
_orig_centers = sorted(
    float(os.path.basename(d).replace("cv_at_", ""))
    for d in glob.glob(f"{ORIG_US_DIR}/cv_at_*")
)
if _orig_centers:
    print(f"Found {len(_orig_centers)} original windows: "
          f"{_orig_centers[0]:.2f} ... {_orig_centers[-1]:.2f}")
else:
    print(f"No original US windows found in {ORIG_US_DIR}. "
          "Falling back to initial_rc_*.xyz files.")

def find_initial_structure(cv_at):
    """Return the path of the initial structure file closest to cv_at.

    Priority:
      1. Equilibrated structure of the original US window (ORIG_US_DIR) with the nearest center
      2. Initial structure extracted in 04 (output/04_initial_structures)
      3. Initial structure shipped in assets (assets/05_umbrella_sampling)
    """
    cv_at_str = f"{cv_at:.2f}"

    # 1. Equilibrated structure of the nearest original US window
    if _orig_centers:
        nearest = min(_orig_centers, key=lambda x: abs(x - cv_at))
        cand = f"{ORIG_US_DIR}/cv_at_{nearest:.2f}/md-dyn-restart.xyz"
        if os.path.exists(cand):
            return cand

    # 2, 3. Initial structures extracted from the SMD trajectory (output first, then assets)
    for cand in [f"./output/04_initial_structures/initial_rc_{cv_at_str}.xyz",
                 f"./assets/05_umbrella_sampling/initial_rc_{cv_at_str}.xyz",
                 f"{ORIG_US_DIR}/initial_rc_{cv_at_str}.xyz"]:
        if os.path.exists(cand):
            return cand

    return None

# Check the sources
for cv in cv_restraint_extra:
    print(f"cv_at={cv:.2f} <- {find_initial_structure(cv)}")

## Step 4. Function that runs the additional umbrella sampling

This is almost the same as `run_us_window` in `05_umbrella_sampling_en.ipynb`, with the following differences.

- Uses the strong value `KAPPA_STRONG` for the spring constant `KAPPA`
- Obtains the initial structure from `find_initial_structure`
- Writes output under `OUT_BASE` (so the original results are not overwritten)
- In addition to `COLVAR`, saves the `KAPPA` value to `meta.txt` so that the subsequent MBAR step can read it

In [ ]:
# ============================================================
# 4. Parallel Function Definition
# ============================================================
def run_us_window_strong(cv_at):
    """Run additional umbrella sampling with a strong spring constant at the given reaction coordinate cv_at"""
    s_time = perf_counter()
    cv_at_str = f"{cv_at:.2f}"

    # 1. Output directory
    out_dir = f"{OUT_BASE}/cv_at_{cv_at_str}"
    os.makedirs(out_dir, exist_ok=True)

    # 2. Calculator (created inside the function to avoid parallel-execution issues)
    estimator = Estimator(
        calc_mode=CALC_MODE,
        method_type=METHOD_TYPE,
        model_version=MODEL_VERSION,
    )
    calculator = ASECalculator(estimator)

    # 3. Atoms & Constraints
    input_xyz = find_initial_structure(cv_at)
    if input_xyz is None:
        return f"Error: Input structure not found for cv_at={cv_at_str}"
    atoms = read(input_xyz)

    # Fix the 1st and 2nd Cu layers (z < FIX_THRESH)
    constraint = FixAtoms(mask=atoms.positions[:, 2] < FIX_THRESH)
    atoms.set_constraint(constraint)

    # 4. PLUMED Settings (strong spring constant)
    plumed_setting = [
        "UNITS LENGTH=A ENERGY=eV",
        f"pos: POSITION ATOM={PULLED_ATOM}",
        f"restraint-z: RESTRAINT ARG=pos.z KAPPA={KAPPA_STRONG} AT={cv_at_str}",
        f"PRINT STRIDE={LOG_INTERVAL} ARG=pos.z,restraint-z.bias,restraint-z.force2 "
        f"FILE={out_dir}/COLVAR_{cv_at_str}",
        "FLUSH STRIDE=1000",
    ]
    atoms.calc = Plumed(
        calc=calculator,
        input=plumed_setting,
        timestep=TIMESTEP,
        atoms=atoms,
        kT=units.kB * TEMPERATURE,
    )

    # 5. MD Setup
    MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE, force_temp=True)
    Stationary(atoms)
    dyn = Langevin(
        atoms,
        TIMESTEP,
        temperature_K=TEMPERATURE,
        friction=0.002 / units.fs,
        trajectory=f"{out_dir}/md-dyn.traj",
        logfile=f"{out_dir}/md-dyn.log",
        loginterval=LOG_INTERVAL,
        fixcm=False,
    )

    # 6. Run
    try:
        # Save metadata so the subsequent MBAR step can read the per-window spring constant
        with open(f"{out_dir}/meta.txt", "w") as f:
            f.write(f"center {cv_at_str}\n")
            f.write(f"kappa {KAPPA_STRONG}\n")
            f.write(f"temperature {TEMPERATURE}\n")
            f.write(f"init_structure {input_xyz}\n")

        dyn.run(TOTAL_STEPS)
        write(f"{out_dir}/md-dyn-restart.xyz", atoms)
        elapsed = perf_counter() - s_time
        return f"Done: cv_at={cv_at_str} ({elapsed:.1f} sec)"
    except Exception as e:
        return f"Failed: cv_at={cv_at_str} Error: {e}"

## Step 5. Run the additional umbrella sampling in parallel

In [ ]:
# ============================================================
# 5. Main Execution
# ============================================================
print(f"Start Parallel Calculation: {len(cv_restraint_extra)} extra windows")
print(f"Settings: n_jobs={N_JOBS}, backend={BACKEND}, KAPPA={KAPPA_STRONG} eV")

results = Parallel(n_jobs=N_JOBS, verbose=VERBOSE, backend=BACKEND)(
    delayed(run_us_window_strong)(cv) for cv in cv_restraint_extra
)

In [ ]:
# Show the results
print("\n--- Results ---")
for res in results:
    print(res)

## Step 6. Check the histograms of the extra windows (overlap check)

We check whether the added strongly-restrained windows cover the region around 12 A sufficiently and with adequate overlap between neighboring windows.
If the overlap is insufficient, either reduce the spacing of `cv_restraint_extra` further or slightly weaken `KAPPA_STRONG`.

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = "retina"

fig, ax = plt.subplots(figsize=(7, 4))
for cv in cv_restraint_extra:
    f = f"{OUT_BASE}/cv_at_{cv:.2f}/COLVAR_{cv:.2f}"
    if not os.path.exists(f):
        continue
    z = np.loadtxt(f)[1:, 1]
    hist, bins = np.histogram(z, bins=15, density=True)
    ctr = 0.5 * (bins[:-1] + bins[1:])
    ax.plot(ctr, hist, label=f"{cv:.2f}")

ax.set_xlabel("Reaction coordinate z (A)")
ax.set_ylabel("Probability density")
ax.set_title(f"Extra windows (KAPPA={KAPPA_STRONG} eV)")
ax.legend(fontsize=8, ncol=2)
ax.grid(True)
plt.show()

## Next Step

In the next notebook, [08_mbar_free_energy_with_extra_en.ipynb](./08_mbar_free_energy_with_extra_en.ipynb),
the original windows (`KAPPA=2.5 eV`) and these extra windows (`KAPPA=10.0 eV`) are analyzed together with MBAR
to recompute the unbiased free energy profile (PMF).
MBAR correctly handles different spring constants and restraint centers per window, so mixing the two is not a problem.